# Data Cleaning

In [184]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

sns.set_style("whitegrid")

## Load Data

In [185]:
df = pd.read_csv("../data/raw/training_set_DM.csv")

# Initial Cleaning and Remove of Leakage Features

In [186]:
print(df.shape)
df.head()

(1048575, 54)


,srch_id,date_time,site_id,visitor_location_country_id,visitor_hist_starrating,visitor_hist_adr_usd,prop_country_id,prop_id,prop_starrating,prop_review_score,...,comp6_rate_percent_diff,comp7_rate,comp7_inv,comp7_rate_percent_diff,comp8_rate,comp8_inv,comp8_rate_percent_diff,click_bool,gross_bookings_usd,booking_bool
0,1,4/4/2013 8:32,12,187,NaN,NaN,219,893,3,3.5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
1,1,4/4/2013 8:32,12,187,NaN,NaN,219,10404,4,4.0,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
2,1,4/4/2013 8:32,12,187,NaN,NaN,219,21315,3,4.5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
3,1,4/4/2013 8:32,12,187,NaN,NaN,219,27348,2,4.0,...,NaN,NaN,NaN,NaN,-1.0,0.0,5.0,0,NaN,0
4,1,4/4/2013 8:32,12,187,NaN,NaN,219,29604,4,3.5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0


In [187]:
# Leakage Features
df = df.drop(columns=[
    'gross_bookings_usd'
])

# Missing Value Handling

In [188]:
# A - prop_location_score 2

df['prop_location_score2_missing'] = (
    df['prop_location_score2']
    .isnull()
    .astype(int)
)

df['prop_location_score2'] = (
    df['prop_location_score2']
    .fillna(-1)
)

- Missing values replaced with `-1`
- Added binary missing-value flag
- `-1` was chosen because valid scores are positive, making missing values distinguishable

In [189]:
#B - Visitor history features

df['visitor_hist_missing'] = (
    df['visitor_hist_starrating']
    .isnull()
    .astype(int)
)

df['visitor_hist_starrating'] = (
    df['visitor_hist_starrating']
    .fillna(0)
)

df['visitor_hist_adr_usd'] = (
    df['visitor_hist_adr_usd']
    .fillna(0)
)

- Missing values replaced with `0`
- Added binary missing-value flag
- Missing values likely indicate that the user has no previous booking history

| value | flag |
| ----- | ---- |
| 4.5   | 0    |
| 0     | 1    |
| 3.0   | 0    |
| 0     | 1    |

0 could mean:
- real value
- OR missing

In [190]:
df['affinity_missing'] = (
    df['srch_query_affinity_score']
    .isnull()
    .astype(int)
)

df['srch_query_affinity_score'] = (
    df['srch_query_affinity_score']
    .fillna(-999)
)

- Missing values replaced with `-999`
- Added binary missing-value flag
- Extreme negative value used to clearly separate missing search-affinity information from valid scores

In [191]:
# Competitor-related columns
comp_rate_cols = [
    col for col in df.columns
    if col.startswith("comp") and col.endswith("_rate")
]

comp_inv_cols = [
    col for col in df.columns
    if col.startswith("comp") and col.endswith("_inv")
]

comp_percent_cols = [
    col for col in df.columns
    if col.startswith("comp") and col.endswith("_rate_percent_diff")
]

# 1. Missing flags: preserve whether competitor data was originally missing
for col in comp_rate_cols + comp_inv_cols + comp_percent_cols:
    df[col + "_missing"] = df[col].isnull().astype(int)

# 2. Fill missing values

# comp*_rate:
# -1 = Expedia more expensive
#  0 = same price
#  1 = Expedia cheaper
# NaN = no competitive data
# Fill NaN with 0 as neutral, but missing flag keeps the original missing information.
for col in comp_rate_cols:
    df[col] = df[col].fillna(0)

# comp*_inv:
# 1 = competitor has no availability
# 0 = both Expedia and competitor have availability
# NaN = no competitive data
# Fill NaN with 0 as neutral, but missing flag keeps the original missing information.
for col in comp_inv_cols:
    df[col] = df[col].fillna(0)

# comp*_rate_percent_diff:
# numeric absolute percentage difference
# NaN = no competitive data
# Fill NaN with 0, while missing flag indicates no competitor price comparison existed.
for col in comp_percent_cols:
    df[col] = df[col].fillna(0)
    
# Number of competitors with available price comparison
df["num_competitors_available"] = (df[[col + "_missing" for col in comp_rate_cols]] == 0).sum(axis=1)

# Whether any competitor data exists
df["has_competitor_data"] = (df["num_competitors_available"] > 0).astype(int)

# Expedia price competitiveness
df["expedia_vs_competitors"] = df[comp_rate_cols].max(axis=1)

# Average competitor price comparison
df["comp_rate_mean"] = df[comp_rate_cols].mean(axis=1)

# Maximum percentage difference with competitors
df["comp_percent_diff_max"] = df[comp_percent_cols].max(axis=1)

# Average percentage difference with competitors
df["comp_percent_diff_mean"] = df[comp_percent_cols].mean(axis=1)

# Outlier Handling

In [192]:
# p99.9 clipping

print(df['price_usd'].max())

upper_price = df['price_usd'].quantile(0.999)

df['price_usd'] = df['price_usd'].clip(
    upper=upper_price
)

print(df['price_usd'].max())

11818011.0
2198.598200000066


# Feature Engineering

In [193]:
# mean price per search

df['price_mean_search'] = (
    df.groupby('srch_id')['price_usd']
    .transform('mean')
)

In [194]:
# relative price

df['price_relative'] = (
    df['price_usd'] /
    df['price_mean_search']
)

In [195]:
# log price
df['price_log'] = np.log1p(df['price_usd'])

In [196]:
# cheapest ranking

df['price_rank'] = (
    df.groupby('srch_id')['price_usd']
    .rank(method='dense')
)

In [197]:
# overall rating

df['rating_review_product'] = (
    df['prop_starrating'] *
    df['prop_review_score']
)

In [198]:
# high rating flag

df['high_rating'] = (
    df['prop_starrating'] >= 4
).astype(int)

In [199]:
# promotion flag variable

In [200]:
# type of stay

df['is_family'] = (
    df['srch_children_count'] > 0
).astype(int)

df['is_long_stay'] = (
    df['srch_length_of_stay'] > 3
).astype(int)

# Competitors

In [201]:
comp_rate_cols = [
    col for col in df.columns
    if col.startswith('comp')
    and col.endswith('_rate')
]

In [202]:
df['num_competitors_available'] = (
    df[comp_rate_cols]
    .notnull()
    .sum(axis=1)
)

In [203]:
df['expedia_vs_competitors'] = (
    df[comp_rate_cols]
    .max(axis=1)
)

In [204]:
df.isnull().sum().sort_values(ascending=False).head(20)

orig_destination_distance    337001
rating_review_product          1484
prop_review_score              1484
srch_id                           0
comp5_rate_missing                0
comp6_inv_missing                 0
comp5_inv_missing                 0
comp4_inv_missing                 0
comp3_inv_missing                 0
comp2_inv_missing                 0
comp1_inv_missing                 0
comp8_rate_missing                0
comp7_rate_missing                0
comp6_rate_missing                0
comp3_rate_missing                0
comp4_rate_missing                0
comp2_rate_missing                0
comp1_rate_missing                0
affinity_missing                  0
visitor_hist_missing              0
dtype: int64

In [205]:
df.to_csv(
    "../data/processed/train_prepared.csv",
    index=False
)